# Network Momentum em Ações Globais — Notebook Profissional

**Estratégia:** adaptação para ações globais do modelo *Network Momentum* de
Pu, Roberts, Dong e Zohren — [*Network Momentum across Asset Classes*, arXiv:2308.11294](https://arxiv.org/abs/2308.11294).

**O que este notebook faz, de ponta a ponta:** baixa os dados (com cache), verifica a qualidade,
constrói as 8 features de momentum (Eqs. 1–3), aprende grafos dinâmicos (Eq. 4), agrega por
ensemble (Eq. 5), normaliza (Eq. 6), propaga o momentum pela rede (Eq. 7), ajusta a OLS
transversal (Eq. 8), monta o portfólio long-short com volatility targeting (Eq. 9), aplica
custos reais por bolsa sobre o turnover efetivo (Eqs. 13–14), compara com benchmarks
metodológicos e externos, roda a suíte de validação estatística e de robustez, gera todos os
gráficos (300 dpi, PNG+SVG+CSV) e produz as respostas do formulário.

> **Honestidade metodológica** — três avisos que valem para tudo o que segue:
> 1. Esta é uma **adaptação** (ações via Yahoo Finance), não uma replicação do artigo
>    (64 futuros da Pinnacle, 1990–2022). Os números não são comparáveis aos do paper.
> 2. O universo atual é estático e contém apenas sobreviventes — há **viés de sobrevivência**
>    não corrigível com dados gratuitos; a magnitude é explorada por ablações.
> 3. Todos os resultados são separados em treino / validação / teste; nada aqui usa
>    informação futura na data do sinal (testado automaticamente).


## Como usar

| Modo | O que faz | Quando usar |
|---|---|---|
| **A — Autônomo** | Clona/usa o ZIP do projeto, instala dependências, baixa dados e roda tudo | Primeira execução, ambiente descartável |
| **B — Google Drive** | Monta o Drive e guarda cache de dados + resultados lá | Continuar experimentos sem novo download |

Perfis de execução (célula de parâmetros abaixo):

- `smoke` — dados **sintéticos**, sem internet, ~1–2 min; valida o encanamento inteiro;
- `fast` — dados reais, pula as ablações mais caras (lookback e arestas), ~30–60 min no Colab;
- `full` — tudo, inclusive ablações de lookback e intra/inter região — pode levar algumas horas.

Execute as células **em ordem**. Todas as saídas ficam em `outputs/<perfil>/`.


In [ ]:
# =============================== PARÂMETROS ===============================
PROFILE = "full"           # "smoke" | "fast" | "full"
USE_DRIVE = False           # Modo B: cache e resultados no Google Drive
GIT_URL = ""                # opcional: URL do repositório para clonar no Modo A
CONFIG_FILE = "config/default.toml"  # ou "config/paper_structure.toml"
RANDOM_SEED = 42            # usado por bootstrap/permutação/PBO (config [validation])


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Ambiente: {'Google Colab' if IN_COLAB else 'local'} | Python {sys.version.split()[0]}")

DRIVE_ROOT = None
if IN_COLAB and USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/network_momentum")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Modo B ativo — cache/resultados em {DRIVE_ROOT}")

def _find_project() -> Path | None:
    candidates = [Path.cwd(), *Path.cwd().parents[:2], Path("/content")]
    if DRIVE_ROOT is not None:
        candidates.append(DRIVE_ROOT)
    for base in candidates:
        if not base.exists():
            continue
        try:
            for pyproject in base.glob("**/pyproject.toml"):
                if not pyproject.is_file():
                    continue
                try:
                    if "network-momentum-yfinance" in pyproject.read_text(encoding="utf-8"):
                        return pyproject.parent
                except OSError as e:
                    print(f"Warning: Could not read {pyproject} due to: {e}", file=sys.stderr)
                    continue
        except OSError as e:
            if "Invalid argument" in str(e) or "Permission denied" in str(e):
                print(f"Warning: Skipping glob for {base} due to: {e}", file=sys.stderr)
                continue
            else:
                raise
    return None

PROJECT_DIR = _find_project()
if PROJECT_DIR is None and GIT_URL:
    subprocess.run(["git", "clone", GIT_URL, "/content/network_momentum_repo"], check=True)
    PROJECT_DIR = _find_project()
if PROJECT_DIR is None and IN_COLAB:
    print("Projeto não encontrado. Envie o ZIP do projeto (Modo A):")
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            shutil.unpack_archive(name, "/content/network_momentum_zip")
    PROJECT_DIR = _find_project()
assert PROJECT_DIR is not None, "Projeto não localizado. Defina GIT_URL ou envie o ZIP."
os.chdir(PROJECT_DIR)
print(f"Projeto: {PROJECT_DIR}")


In [ ]:
# ============ REGISTRO DA EXECUÇÃO (data, config, seed, ambiente) ============
import datetime as dt, dataclasses, json
from pathlib import Path
import os, sys, subprocess, importlib, site # Added imports

# --- Start of added workaround ---
# Workaround for ModuleNotFoundError if network_momentum wasn't installed correctly
try:
    import network_momentum
    importlib.reload(network_momentum) # Ensure any updates are picked up
except ImportError:
    print("Module 'network_momentum' not found. Attempting to install it now...", file=sys.stderr)
    # PROJECT_DIR should be available from a previous cell's execution
    _project_dir = globals().get('PROJECT_DIR')
    if _project_dir is None:
        raise RuntimeError("PROJECT_DIR global variable is not defined. Cannot proceed.")

    # Explicitly pass cwd to subprocess.run to ensure pip installs from the correct location
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[dev]"],
        capture_output=True, text=True,
        cwd=str(_project_dir)
    )
    if install_result.returncode != 0:
        print("Error during network_momentum installation:", file=sys.stderr)
        print(install_result.stdout, file=sys.stderr)
        print(install_result.stderr, file=sys.stderr)
        raise RuntimeError(f"Failed to install 'network_momentum' from {_project_dir}")
    print("'network_momentum' installed successfully. Reloading module paths.", file=sys.stderr)
    
    # Refresh site-packages to pick up the new editable installation
    site.main()
    
    # Explicitly add the project directory (and potentially src/) to sys.path
    if str(_project_dir) not in sys.path:
        sys.path.insert(0, str(_project_dir))
    _src_dir = _project_dir / "src"
    if _src_dir.exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        
    importlib.invalidate_caches() # Ensure Python picks up the new module
    # Try importing again after successful installation
    import network_momentum # This import should now succeed
# --- End of added workaround ---

from network_momentum.config import load_config
from network_momentum.universe import load_universe, universe_fingerprint

config = load_config(CONFIG_FILE)

# Redireciona cache/saída para o Drive no Modo B e isola a saída por perfil.
output_base = (DRIVE_ROOT / "outputs" if DRIVE_ROOT else Path("outputs")) / PROFILE
if DRIVE_ROOT is not None:
    config = dataclasses.replace(
        config, data=dataclasses.replace(config.data, cache_dir=DRIVE_ROOT / "data_cache")
    )
if RANDOM_SEED != config.validation.seed:
    config = dataclasses.replace(
        config, validation=dataclasses.replace(config.validation, seed=RANDOM_SEED)
    )

universe = load_universe(config.data.universe_path)
print(json.dumps({
    "data_execucao_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "configuracao": str(config.source_path),
    "perfil": PROFILE,
    "periodo_solicitado": f"{config.data.start} → {config.data.end or 'hoje'}",
    "universo": f"{len(universe)} ativos, {universe['region'].nunique()} regiões",
    "hash_universo": universe_fingerprint(universe),
    "moeda_base": config.data.base_currency,
    "seed": config.validation.seed,
    "saida": str(output_base),
}, indent=2, ensure_ascii=False))


In [ ]:
# ============ REGISTRO DA EXECUÇÃO (data, config, seed, ambiente) ============
import datetime as dt, dataclasses, json
from pathlib import Path
from network_momentum.config import load_config
from network_momentum.universe import load_universe, universe_fingerprint

config = load_config(CONFIG_FILE)

# Redireciona cache/saída para o Drive no Modo B e isola a saída por perfil.
output_base = (DRIVE_ROOT / "outputs" if DRIVE_ROOT else Path("outputs")) / PROFILE
if DRIVE_ROOT is not None:
    config = dataclasses.replace(
        config, data=dataclasses.replace(config.data, cache_dir=DRIVE_ROOT / "data_cache")
    )
if RANDOM_SEED != config.validation.seed:
    config = dataclasses.replace(
        config, validation=dataclasses.replace(config.validation, seed=RANDOM_SEED)
    )

universe = load_universe(config.data.universe_path)
print(json.dumps({
    "data_execucao_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "configuracao": str(config.source_path),
    "perfil": PROFILE,
    "periodo_solicitado": f"{config.data.start} → {config.data.end or 'hoje'}",
    "universo": f"{len(universe)} ativos, {universe['region'].nunique()} regiões",
    "hash_universo": universe_fingerprint(universe),
    "moeda_base": config.data.base_currency,
    "seed": config.validation.seed,
    "saida": str(output_base),
}, indent=2, ensure_ascii=False))


## 1. A ideia econômica, sem fórmulas

**O que é momentum?** A observação empírica de que ativos que subiram (ajustado pelo risco)
nos últimos meses tendem, *em média*, a continuar subindo no curto prazo — e vice-versa. É um
dos fatores mais documentados da literatura (Jegadeesh & Titman, 1993; Moskowitz, Ooi &
Pedersen, 2012).

**O que é momentum spillover?** O momentum de um ativo também prevê o retorno de ativos
**economicamente ligados** a ele: fornecedores e clientes, empresas do mesmo setor, países
com laços comerciais. A informação viaja devagar entre ativos — investidores de Vale não
reagem instantaneamente ao que aconteceu com as mineradoras australianas.

**Por que representar os ativos como uma rede?** Porque "ligação econômica" é uma relação
entre pares. Um **nó** é um ativo (uma ação). Uma **aresta** é a força da similaridade entre
os históricos de momentum de dois ativos — se os oito sinais de momentum de duas ações se
movem juntos há anos, o modelo cria uma aresta forte entre elas. Em vez de exigir dados de
cadeias produtivas (caros e incompletos), o artigo **aprende o grafo diretamente dos preços**,
resolvendo um problema de otimização convexa (Eq. 4) que exige: pesos **simétricos**
(a relação A→B é a mesma que B→A), **não negativos** (arestas medem afinidade, não sinal) e
**sem autoarestas** (a diagonal zero impede que o ativo "espie" o próprio momentum duas vezes).

**O papel de α e β:** os dois hiperparâmetros controlam a topologia. O termo `−α·log(grau)`
impede nós isolados (todo ativo mantém ao menos uma conexão); o termo `β‖A‖²` espalha os
pesos e controla a esparsidade. Valores menores → grafo mais esparso → cada ativo escuta
poucos vizinhos; valores maiores → grafo denso → o sinal de rede vira uma média de mercado.
São escolhidos por busca em grade **apenas na validação** (últimos 10% do treino).

**Por que vários lookbacks (252…1260 dias)?** Um grafo estimado com 1 ano de dados reage
rápido a mudanças de regime, mas é ruidoso; um com 5 anos é estável, mas lento. O ensemble
(média das adjacências, Eq. 5) reduz a variância das arestas — o artigo mostra que também
melhora o resultado e reduz o turnover.

**Como o sinal vira posição?** As features dos vizinhos são propagadas (Eq. 7: média
ponderada pelas arestas normalizadas), uma única OLS transversal (Eq. 8) transforma as 8
features de rede em uma previsão do retorno de amanhã ajustado por volatilidade, e a posição
é o **sinal** da previsão: prevista alta → comprado; prevista queda → vendido. O peso de cada
ativo é `σ_alvo/σ_ativo` (Eq. 9): ativos mais voláteis recebem menos capital, igualando a
contribuição de risco. Uma segunda camada de **volatility targeting** no portfólio ajusta a
alavancagem para manter ~15% a.a. de volatilidade — com teto de alavancagem e estimador
defasado (sem olhar o futuro).

**Por que o turnover destrói parte do resultado?** A estratégia decide posições todo dia.
Cada mudança de peso paga spread, corretagem, emolumentos e, em várias bolsas do universo,
tributos (0,5% na compra no Reino Unido; 0,1% por lado em Hong Kong…). O retorno líquido é
`retorno bruto − custo × turnover` (Eq. 14) — por isso reportamos break-even e a curva
Sharpe × custo.


## 2. Protocolo temporal — onde mora a honestidade do backtest

```
2005 ──────────── 2014 | 2015 ──── 2019 | 2020 ──── 2024 | 2025 ──
       treino 1        |    teste 1     |                |
       treino 2 (expandido)             |    teste 2     |
       treino 3 (expandido)                              |  teste 3
```

- **Walk-forward expansivo**: o modelo é reestimado a cada 5 anos usando só o passado.
- **Validação interna (nested)**: os últimos 10% de cada treino escolhem (α, β); o teste
  nunca participa de nenhuma escolha.
- **Embargo** de 1 pregão entre treino e teste (o alvo é o retorno de t+1).
- **Signal lag de 1 pregão** (config padrão): as bolsas fecham em horários diferentes —
  usar o fechamento de Tóquio "de hoje" para operar em NY "hoje" seria viável, mas usar o
  contrário não; o lag de 1 dia torna tudo executável em qualquer fuso (o artigo, com
  futuros no mesmo fechamento, usa lag 0 — perfil `paper_structure.toml`).
- Features usam **apenas** preços ≤ t−lag; o alvo usa apenas t→t+1; testado em
  `tests/test_features.py` e `tests/test_portfolio.py`.


In [ ]:
# ===================== TESTES RÁPIDOS ANTES DO PIPELINE =====================
# A suíte cobre: ausência de look-ahead, alvo deslocado, restrições do grafo,
# turnover (Eq. 13), custos por lado, FX, alavancagem causal, splits/embargo etc.
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-m", "not slow", "-q", "--no-header"],
    capture_output=True, text=True, cwd=str(PROJECT_DIR),
)
print(result.stdout[-3000:])
print(result.stderr[-1000:])
assert result.returncode == 0, "Testes rápidos falharam — não prossiga."


In [ ]:
# ========================= EXECUÇÃO DO PIPELINE =========================
# Tudo (dados → features → grafos → backtest → benchmarks → custos → validação
# → robustez → topologia → gráficos → tabelas → manifest → formulário) sai de
# UMA função — a mesma usada pelo CLI (`python -m network_momentum --profile ...`),
# garantindo consistência notebook ↔ módulos.
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

from network_momentum.pipeline import PipelineOptions, run_full_pipeline

options = PipelineOptions(
    profile=PROFILE,
    output_dir=output_base,
    make_plots=True,
    run_lookback_ablation_flag=(PROFILE == "full"),
    run_edge_ablation_flag=(PROFILE == "full"),
    run_regression_variants=(PROFILE != "smoke"),
)
artifacts = run_full_pipeline(config, options)
print("\nSaídas em:", artifacts["output_dir"])


In [ ]:
# ==================== MANIFESTO DA EXECUÇÃO (reprodutibilidade) ====================
import json
manifest = artifacts["manifest"]
print(json.dumps({k: manifest[k] for k in (
    "generated_at_utc", "platform", "in_colab", "library_versions", "base_currency",
    "seed", "universe_hash", "n_assets", "oos_start", "oos_end", "folds",
    "git_commit", "profile", "net_sharpe", "breakeven_bps", "runtime_seconds",
) if k in manifest}, indent=2, ensure_ascii=False))


## 3. Resultados principais

A tabela abaixo compara a estratégia principal (**GMOM**, momentum de rede) com os
benchmarks metodológicos — todos no **mesmo universo, mesmo período, mesma moeda, mesma
Eq. (9) e mesmo motor de custos**:

- `linreg` — Eq. (11), as mesmas 8 features **sem** a rede: é o teste direto de
  "a rede adiciona valor?";
- `regcombo`/`signcombo` — combinações de momentum individual e de rede (Eq. 12);
- `macd` — Eq. (10), momentum clássico sem regressão;
- `long_only` (vol-scaled) e `equal_weight` — o mercado do universo.

**Como ler:** GMOM só se justifica se superar `linreg` (senão a rede não agrega) e se
sobreviver aos custos (coluna líquida vs bruta). Ambas as comparações estão aqui.


In [ ]:
import pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
tables = artifacts["output_dir"] / "tables"

print("=== Métricas LÍQUIDAS por estratégia (custos reais, cenário base) ===")
display(pd.read_csv(tables / "strategies_net_metrics.csv", index_col=0)
        [["annual_return", "cagr", "annual_volatility", "sharpe", "sortino", "calmar",
          "max_drawdown", "hit_rate", "skewness", "var_95"]])

print("=== Métricas BRUTAS (antes de custos) ===")
display(pd.read_csv(tables / "strategies_gross_metrics.csv", index_col=0)
        [["annual_return", "annual_volatility", "sharpe", "max_drawdown"]])

print("=== Operacional (GMOM): turnover, custos, exposições, alavancagem ===")
display(pd.read_csv(tables / "gmom_operational.csv", index_col=0))

print("=== Folds do walk-forward: Sharpe treino vs validação vs teste ===")
folds = pd.read_csv(tables / "gmom_folds.csv", index_col=0)
fold_metrics = pd.read_csv(tables / "gmom_metrics_by_fold.csv", index_col=0)
comparison = folds[["train_start", "test_start", "test_end", "alpha", "beta",
                    "train_sharpe", "validation_sharpe"]].copy()
comparison["test_sharpe"] = fold_metrics["sharpe"]
display(comparison)


In [ ]:
from IPython.display import Image, display as show
figures = artifacts["output_dir"] / "figures"
for name in ("01_equity_gross_net", "02_equity_vs_benchmarks", "03_drawdown",
             "05_rolling_sharpe", "08_turnover"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=880))


## 4. Custos — de onde vêm e quanto destroem

O custo de cada operação é decomposto por componente e por bolsa
(`config/costs.csv`, com fonte, data de acesso e a distinção **oficial vs estimativa**):

1. corretagem (estimativa institucional); 2. emolumentos da bolsa; 3. clearing;
4. taxas regulatórias (ex.: SEC fee na venda); 5. meio bid-ask spread (3 cenários);
6. slippage/impacto (lei de raiz quadrada, com limite de participação no ADV);
7. **tributos** — SDRT UK 0,5% na compra (oficial, gov.uk), stamp HK 0,1%/lado (oficial,
gov.hk), STT Índia 0,1%/lado e FTT França (marcadas para verificação); 8. conversão cambial;
9. aluguel (borrow) em posições vendidas; 10. custo da variação de alavancagem — o turnover
usa o **peso final**, incluindo o efeito do volatility targeting.

**Interpretação do break-even:** é o custo linear (bps por unidade de turnover) que zeraria
o retorno médio. Se os custos reais estimados da sua execução ficarem acima dele, a
estratégia não sobrevive — não há ajuste fino que salve.


In [ ]:
print("=== Cenários de custo (conservador / base / otimista) ===")
display(pd.read_csv(tables / "cost_scenarios.csv", index_col=0)
        [["annual_return", "sharpe", "max_drawdown", "annual_cost_real"]])

print(f"Break-even (custo linear que zera o retorno): {artifacts['breakeven_bps']:.1f} bps")
print("=== Sharpe líquido × custo (Eq. 14): 0 a 10 bps ===")
display(pd.read_csv(tables / "sharpe_vs_cost.csv", index_col=0))

capacity_path = tables / "capacity_estimate.csv"
if capacity_path.exists():
    print("=== Capacidade estimada (ativos mais restritivos primeiro) ===")
    display(pd.read_csv(capacity_path, index_col=0).head(10))

from IPython.display import Image, display as show
for name in ("09_cost_decomposition", "10_sharpe_vs_cost"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=880))


## 5. Moeda-base e câmbio

Os retornos existem em **duas versões**, nunca misturadas em silêncio:

- **Local (proxy hedgeada)** — cada ativo na própria moeda. Aproxima uma carteira com hedge
  cambial, mas **ignora o custo do hedge** (o diferencial de juros dos forwards). Para
  moedas de juro alto (BRL, INR), o hedge real custaria vários % ao ano — esta versão é,
  portanto, um **limite superior** da versão hedgeada.
- **USD sem hedge** — preços convertidos pelo câmbio do dia (pares do Yahoo), com o trecho
  cambial cobrindo exatamente a mesma janela do retorno do ativo (importante em feriados
  locais). A diferença entre as duas séries é o efeito cambial realizado da carteira.

Não fazer hedge significa carregar, junto do momentum, uma exposição comprada nas moedas dos
ativos comprados e vendida nas dos vendidos — em crises, tipicamente um beta comprado em
moedas cíclicas contra USD.


In [ ]:
fx_path = tables / "currency_comparison.csv"
if fx_path.exists():
    print("=== Local (proxy hedgeada) vs USD sem hedge ===")
    display(pd.read_csv(fx_path, index_col=0)
            [["annual_return", "annual_volatility", "sharpe", "max_drawdown"]])
else:
    print("Comparação cambial indisponível (perfil smoke usa universo 100% USD).")

ext_path = tables / "external_benchmarks.csv"
if ext_path.exists():
    print("=== Contra benchmarks externos (alpha a.a., beta, correlação, TE, IR, capturas) ===")
    display(pd.read_csv(ext_path, index_col=0))


## 6. Overfitting — como estamos tentando nos enganar e não conseguindo

Cada diagnóstico ataca uma forma diferente de auto-engano. **Nenhum deles, isolado, prova
ausência de overfitting** — e um teste OOS positivo também não.

| Diagnóstico | Pergunta que responde | Como ler |
|---|---|---|
| Treino vs validação vs teste | O desempenho degrada fora do treino? | Quedas grandes (>50%) de Sharpe = memorização |
| Learning curve | Mais dados ajudam? | Sharpe de validação crescendo com a fração de treino = modelo aprende estrutura real |
| Bootstrap em blocos (por data) | O Sharpe é distinguível de zero dada a autocorrelação? | IC 95% contendo 0 = evidência fraca |
| Permutação (deslocamento circular) | E se o alinhamento sinal→retorno fosse aleatório? | p-valor alto = o "skill" pode ser acaso |
| Deflated Sharpe (DSR) | O Sharpe sobrevive ao nº de configurações tentadas? | DSR < 0,95 = ceticismo |
| PBO / CSCV | O vencedor in-sample continua vencedor out-of-sample? | PBO > 30–40% = seleção frágil |
| Estabilidade dos coeficientes | O modelo muda de ideia a cada fold? | Trocas de sinal frequentes = sinal fraco |
| Sensibilidade a hiperparâmetros | O resultado depende de um ponto mágico do grid? | Vizinhos do grid com desempenho similar = robusto |

**Correção de múltiplos testes** (Bonferroni/Benjamini-Hochberg) é aplicada aos p-valores da
família de testes acima.


In [ ]:
bootstrap = artifacts["bootstrap"]
print(f"Bootstrap ({config.validation.bootstrap_samples} amostras, blocos de "
      f"{config.validation.bootstrap_block_days} dias, seed {config.validation.seed}):")
print(f"  Sharpe observado: {bootstrap.observed_sharpe:.3f}")
print(f"  IC 95%: [{bootstrap.ci_low:.3f}, {bootstrap.ci_high:.3f}]")
print(f"  P(Sharpe <= 0): {bootstrap.p_value_positive:.4f}")

permutation = artifacts.get("permutation")
if permutation is not None:
    print(f"\nPermutação por deslocamento circular ({len(permutation.permuted_sharpes)} amostras):")
    print(f"  p-valor: {permutation.p_value:.4f}")

print("\nDeflated Sharpe Ratio (Bailey & López de Prado):")
display(pd.read_csv(tables / "deflated_sharpe.csv", index_col=0))

pbo = artifacts["pbo"]
if pbo.applicable:
    print(f"PBO (CSCV, {pbo.n_configurations} configurações, {pbo.n_splits} splits): {pbo.pbo:.2%}")
else:
    print(f"PBO: não aplicável — {pbo.note}")

print("\nCorreção de múltiplos testes:")
display(pd.read_csv(tables / "multiple_testing.csv", index_col=0))

print("\nEstabilidade dos coeficientes entre folds (SE clusterizado por data):")
display(pd.read_csv(tables / "coefficient_stability.csv", index_col=0))

from IPython.display import Image, display as show
for name in ("22_learning_curves", "23_train_val_test", "24_bootstrap_sharpe",
             "25_permutation_test", "26_coefficient_stability"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=820))


## 7. Underfitting — o erro simétrico

Um modelo pode ser honesto e ainda assim fraco demais. Sinais de underfitting e onde olhar:

- **R² minúsculo e igual em treino e teste** — normal em retornos diários (R² na casa de
  0,1–1% já move dinheiro), mas se o Sharpe do sinal em TREINO já for ~0, não há o que
  generalizar (tabela de learning curve);
- **Resíduos com autocorrelação** (Ljung-Box significativo) — estrutura temporal que a OLS
  transversal não captura;
- **Regras simples empatadas** — se `macd` ou `long_only` empatam com GMOM líquido, a
  complexidade não se paga;
- **OLS vs Ridge/Lasso/ElasticNet** — se a regularização *melhora* muito, a OLS estava
  sofrendo com multicolinearidade das 8 features de rede (elas são bastante correlacionadas);
  os métodos regularizados aqui são **testes de robustez**, não substitutos silenciosos;
- **Sinal contínuo vs sign()** — o artigo usa sign(previsão); se a versão contínua ou um
  threshold de convicção mudar muito o resultado, a informação está na magnitude, não só no
  sinal.


In [ ]:
print("=== Diagnóstico dos resíduos (OOS) ===")
display(pd.read_csv(tables / "residual_diagnostics.csv", index_col=0))

print("=== R² univariado por feature de rede ===")
display(pd.read_csv(tables / "univariate_feature_r2.csv", index_col=0))

print("=== Learning curve (frações cronológicas do treino) ===")
display(pd.read_csv(tables / "learning_curve.csv", index_col=0))

reg_path = tables / "regression_variants.csv"
if reg_path.exists():
    print("=== OLS vs Ridge vs Lasso vs Elastic Net ===")
    display(pd.read_csv(reg_path, index_col=0)[["annual_return", "sharpe", "max_drawdown"]])

print("=== Variantes de sinal: sign, thresholds de convicção, contínuo ===")
display(pd.read_csv(tables / "signal_threshold_study.csv", index_col=0)
        [["annual_return", "sharpe", "max_drawdown", "mean_turnover"]])

from IPython.display import Image, display as show
for name in ("27_residual_autocorrelation", "31_signal_threshold", "32_regression_variants"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=820))


## 8. Robustez — onde o resultado mora

Perguntas que as tabelas abaixo respondem:

- **Por região/bolsa/moeda/setor** — o retorno vem de todo lugar ou de um único mercado?
- **Long vs short** — a perna vendida contribui ou só custa? (em bull markets longos, o
  short de momentum costuma perder dinheiro e "pagar o seguro");
- **Contribuição e concentração** — remover os 1–5 melhores ativos derruba o Sharpe?
  Universos com viés de sobrevivência tendem a concentrar nos vencedores óbvios;
- **Remoção dos melhores anos / datas iniciais alternativas** — o resultado é um par de
  anos bons ou um fluxo constante?
- **Regimes de volatilidade e janelas de crise** — momentum clássico sofre em reversões
  bruscas (momentum crash); a versão de rede também?
- **Ablações de features/arestas/lookbacks** — cada peça contribui ou é peso morto?


In [ ]:
for name, title in [
    ("metrics_by_region", "Por região"),
    ("metrics_by_currency", "Por moeda"),
    ("metrics_by_exchange", "Por bolsa"),
    ("metrics_by_sector", "Por setor (aproximado)"),
    ("long_short_decomposition_metrics", "Long vs Short"),
    ("remove_best_assets", "Removendo os melhores ativos"),
    ("remove_best_years", "Removendo os melhores anos"),
    ("start_date_sensitivity", "Sensibilidade à data inicial"),
    ("volatility_regimes", "Regimes de volatilidade"),
    ("calendar_windows", "Janelas de crise/eventos"),
    ("feature_ablation", "Ablação de features"),
    ("edge_ablation", "Ablação intra/inter (região e setor)"),
    ("lookback_ablation", "Lookback individual vs ensemble"),
    ("return_concentration", "Concentração de retorno"),
    ("risk_concentration", "Concentração de risco"),
]:
    path = tables / f"{name}.csv"
    if path.exists():
        frame = pd.read_csv(path, index_col=0)
        if not frame.empty:
            print(f"=== {title} ===")
            columns = [c for c in ("annual_return", "sharpe", "max_drawdown", "hit_rate",
                                    "value") if c in frame.columns]
            display(frame[columns] if columns else frame)

from IPython.display import Image, display as show
for name in ("11_sharpe_by_region", "15_long_short", "16_asset_contribution",
             "18_strategy_correlation", "19_sign_agreement"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=820))


## 9. Topologia do grafo — o modelo está vendo estrutura real?

Métricas da Seção 5.1 do artigo, calculadas sobre o ensemble com threshold relativo de
aresta (o solver suave não produz zeros exatos — diferença documentada vs o solver convexo
do paper, disponível via `graph.solver = "cvxpy"`):

- **edge sparsity** (densidade) — fração de pares conectados;
- **node degree** — a quantos vizinhos cada ativo escuta;
- **clustering coefficient** — os vizinhos dos meus vizinhos são meus vizinhos?
- **community ratio** — as arestas respeitam as regiões? (no artigo, as classes de ativos);
- **Jaccard entre snapshots consecutivos** — estabilidade temporal; no artigo fica >0,99 e
  cai em 2008/2020 (o grafo se reorganiza em crises).


In [ ]:
display(pd.read_csv(tables / "graph_topology.csv", index_col=0).describe())
from IPython.display import Image, display as show
for name in ("28_graph_normal", "29_graph_crisis", "30_topology_series"):
    path = figures / f"{name}.png"
    if path.exists():
        show(Image(filename=str(path), width=840))


## 10. Limitações — o que este backtest NÃO prova

**Dos dados**
1. **Viés de sobrevivência**: universo estático com 36 vencedores atuais; um universo
   point-in-time exigiria dados pagos (CRSP/Datastream). As ablações de "melhores ativos"
   dimensionam a sensibilidade, não corrigem o viés.
2. **Yahoo Finance**: preços ajustados podem ser revisados; qualidade heterogênea fora dos
   EUA; adequado para pesquisa, não para produção.
3. **HSBC duplicada** (HSBA.L e 0005.HK): mesma empresa em duas bolsas — mantida por decisão
   de universo, documentada; o grafo naturalmente cria aresta forte entre as duas.
4. **Dados diários**: spread/impacto são cenários metodológicos, não medições intradiárias.

**Econômicas**
5. **Custos**: tributos estatutários tornam o breakeven apertado em UK/HK/FR/IN; short em
   NSE é restrito para estrangeiros; borrow estimado.
6. **Execução multi-fuso**: o lag de 1 dia resolve a informação, mas a execução real exige
   mesa 24h ou execução regional escalonada.
7. **Capacidade**: universo pequeno e mega caps → capacidade razoável, mas a estimativa usa
   medianas (grosseira por construção).
8. **Regime**: o período OOS (2015+) não contém um bear market secular de vários anos.


## 11. Melhorias próprias vs artigo (cada uma com hipótese e risco)

| # | Alteração | Hipótese/motivação | Referência | Resultado esperado | Risco | Onde ver o resultado |
|---|---|---|---|---|---|---|
| 1 | Signal lag de 1 pregão | Executabilidade multi-fuso sem look-ahead | prática de mercado | Sharpe ~igual ou menor que lag 0 | nenhum (mais conservador) | comparar perfis default vs paper_structure |
| 2 | Conversão FX + duas visões de retorno | Sem moeda-base o retorno agregado não é interpretável | convenção padrão | diferença local vs USD visível em crises | nenhum | tabela currency_comparison |
| 3 | Custos por bolsa/lado + cenários | 1 bp único subestima tributos estatutários | docs oficiais (gov.uk, gov.hk) | Sharpe líquido menor e realista | nenhum | cost_scenarios, breakeven |
| 4 | Turnover inclui Δalavancagem | Vol targeting também gera trades | Eq. 13 estendida | custo ~maior | nenhum | gmom_operational |
| 5 | Elegibilidade por lookback no grafo | Paper usa disponibilidade por janela δ | §3.1 do artigo | mais ativos nos grafos curtos | leve mudança vs versão antiga | audit/CHANGELOG |
| 6 | Ridge/Lasso/ElasticNet auxiliares | 8 features de rede são colineares | Hastie et al. (2009) | Sharpe similar; coeficientes estáveis | +3 hiperparâmetros (só robustez) | regression_variants |
| 7 | Thresholds de convicção | Trades de baixa convicção pagam custo e não geram retorno | Lim et al. (2019) | menos turnover, Sharpe líquido ≥ | +1 hiperparâmetro (estudo, não default) | signal_threshold_study |
| 8 | Solver CVXPY opcional | Verificar a aproximação do L-BFGS-B | Kalofolias (2016) | arestas com zeros exatos | dependência opcional | graph.solver="cvxpy" |


In [ ]:
# ==================== RESPOSTAS DO FORMULÁRIO (Q9–Q15) ====================
answers = artifacts["form_answers"]
print("========== VERSÃO CURTA (copiar no Microsoft Forms) ==========\n")
for question, answer in answers["curta"].items():
    print(f"{question}\n{answer}\n")
print("\n========== VERSÃO TÉCNICA (relatório) ==========\n")
for question, answer in answers["tecnica"].items():
    print(f"{question}\n{answer}\n")
print(f"Arquivo: {artifacts['output_dir'] / 'form_answers.md'}")


In [ ]:
# ================== CÉLULA FINAL: SUÍTE COMPLETA DE TESTES ==================
# Inclui o teste lento de ponta a ponta (pipeline smoke em dados sintéticos).
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--no-header"],
    capture_output=True, text=True, cwd=str(PROJECT_DIR),
)
print(result.stdout[-4000:])
print(result.stderr[-1500:])
status = "TODOS OS TESTES PASSARAM" if result.returncode == 0 else "HÁ TESTES FALHANDO"
print(f"\n{'='*60}\n{status}\n{'='*60}")

output_dir = artifacts["output_dir"]
n_tables = len(list((output_dir / "tables").glob("*.csv")))
n_figures = len(list((output_dir / "figures").glob("*.png")))
print(f"Artefatos: {n_tables} tabelas CSV, {n_figures} figuras (PNG+SVG+CSV) em {output_dir}")
assert result.returncode == 0


## 12. Classificação final

Com base em tudo acima, a avaliação desta equipe é:

| Estágio | Veredito | Justificativa |
|---|---|---|
| Pronta para pesquisa | **Sim** | Pipeline reproduzível, sem look-ahead detectado, diagnósticos completos |
| Pronta para competição acadêmica | **Sim** | Metodologia fiel ao artigo com adaptações documentadas; relatório auditável |
| Pronta para paper trading | **Parcialmente** | Exige fonte de dados de produção e mesa multi-fuso; o sinal em si é executável |
| Pronta para capital real | **Não** | Viés de sobrevivência não corrigido, universo pequeno, custos parcialmente estimados, OOS sem bear market secular |

A justificativa detalhada, o changelog completo e as limitações estão em
`docs/` (AUDIT.md, CHANGELOG.md, LIMITATIONS.md, FUTURE_WORK.md, REFERENCES.md).
